# PhysIR — Stage 6: SEED REPLICATES @ 256/200 (Kaggle, ONE FLOW)

Seed-robustness for the load-bearing arms, on Kaggle's free quota (2×T4 DDP). Regime is
**byte-identical to the seed-42 Set runs** — crop 256, batch 4/GPU (**eff 8**), lr 2.8e-4
cosine, EMA, 200 ep, ep150 snapshot, atomic saves every 2 ep — so replicate deltas are
directly comparable to the primary table. **Only the training seed changes; the 90/10 data
split stays seed-42 forever.**

**Default = seed 2024** (the extra seed the Colab run won't reach; Colab is doing the 1337
pair). `SEEDS = [2024]` in the config cell is the only knob: use `[1337]`, or `[1337, 2024]`
to do both here. Trim `ARMS` to run fewer arms.

| arm | losses beyond base | why it's a load-bearing replicate |
|---|---|---|
| abl_baseline | — | reference of the headline contrast |
| abl_full | phase 0.20 + retinex 0.18 | the physics contribution |
| abl_percep | LPIPS(VGG19) 0.01 | the physics-vs-perceptual contrast |
| abl_full_percep | physics + LPIPS | the interaction (2×2) null |

**Order is PAIR-MAJOR** — baseline → full → percep → full+percep — so a quota cut always
leaves *complete, comparable pairs* rather than orphaned arms.

⏱️ **Cost reality:** ~8.4 min/ep on T4×2 → **~28 h per arm** (~35 h for the LPIPS arms) →
all four ≈ **125 h**, i.e. 2–3 weekly quota cycles (or split across your two accounts). The
headline pair alone (baseline + full ≈ 56 h) fits one linked-account week and is the single
most valuable result — consider running just that first.

## Session workflow (identical to the Set notebooks)
1. **Accelerator: GPU T4 × 2.** Inputs: `physir-code`, `lolblur-dataset`.
2. Run All; let it train. No self-termination — atomic 2-epoch checkpoints mean a session
   kill loses ≤2 epochs.
3. ⚠️ A resume point exists only if the session's output was **saved as a version**.
4. Next session: **Add Input → Your Work → this notebook** (previous version's output) →
   Run All. Finished arms are skipped; partial arms resume from their checkpoint.
5. Repeat until the leaderboard lists every run; download `metrics_out/per_image/` for the
   cross-seed analysis (per-image CSVs at ep150/best/last, same format as the Set runs).

> Cross-seed reading (offline): per seed, the physics delta is `full_sN − baseline_sN`;
> report mean ± spread across seeds {42, 1337, 2024} alongside the seed-42 paired Wilcoxon.

## 1. Install

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

## 2. Paths + stale-code guard

In [ ]:
import re
import os, sys, glob
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'
WORKING_DIR = '/kaggle/working'
def find_root(base, marker, depth=3):
    if os.path.exists(os.path.join(base, marker)): return base
    for d in range(1, depth+1):
        h = sorted(glob.glob(os.path.join(base, *(['*']*d), marker)))
        if h: return h[0][:-len(marker)].rstrip(os.sep)
    return None
CODE_ROOT = find_root(CODE_DIR, 'train.py'); assert CODE_ROOT
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)
src = open(os.path.join(CODE_ROOT,'train.py')).read()
miss = [n for n in ('restore_phys_state','class ModelEMA','best_psnr',
                    'snapshot_epochs','time_budget_h','Seeded RNGs','_atomic_save') if n not in src]
lsrc = open(os.path.join(CODE_ROOT,'losses','loss.py')).read()
if 'class LPIPSLoss' not in lsrc: miss.append('class LPIPSLoss')
assert not miss, f'STALE physir-code, missing {miss} — re-upload with the current train.py + losses/.'
LOLBLUR = find_root(LOLBLUR_ROOT, os.path.join('train','high_sharp_scaled')) or LOLBLUR_ROOT
cand = find_root(LOLBLUR_ROOT, os.path.join('test','low_blur_noise'))
LOLBLUR = cand or LOLBLUR
# ── ORDERING GUARD: the test loader must not shuffle, or per-image rows become a random
# permutation and every paired statistic built on them is invalid (the 2026-07-30 bug).
_dsrc = open(os.path.join(CODE_ROOT, 'data', 'dataset_reader', 'dataset_LOLBlur.py')).read()
_shuf = [d for d in re.findall(r'DataLoader\([^)]*\)', _dsrc, re.S)
         if 'shuffle=True' in d.replace(' ', '')]
assert not _shuf, ('STALE physir-code: test DataLoader still has shuffle=True — '
                   're-upload physir-code with the shuffle=False fix.')
print('✅ ordering guard passed (test loader is not shuffled)')
print('CODE_ROOT:', CODE_ROOT, '| features present ✅')
print('LOLBLUR  :', LOLBLUR, '| has test/:', os.path.isdir(os.path.join(LOLBLUR,'test')))

## 2b. Session restore (attach previous version's output as an input)

In [ ]:
# Continuation sessions: Add Input → Your Work → this notebook (previous version's
# output), then just Run All. Auto-detects checkpoints/logs in ANY attached input and
# copies them into /kaggle/working — fresh sessions print 'nothing to restore'.
import shutil
def _restore():
    n = 0
    for pat in ('/kaggle/input/*/abl_*.pt', '/kaggle/input/*/*/abl_*.pt',
                '/kaggle/input/*/*/*/abl_*.pt'):
        for src in glob.glob(pat):
            dst = os.path.join(WORKING_DIR, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst); n += 1
    # run logs live in visible metrics_out/ (Kaggle outputs can drop dot-dirs)
    logs = []
    for base in ('metrics_out', '.metrics'):
        logs += glob.glob(f'/kaggle/input/*/{base}/*/*.csv')
        logs += glob.glob(f'/kaggle/input/*/*/{base}/*/*.csv')
        logs += glob.glob(f'/kaggle/input/*/*/*/{base}/*/*.csv')
    for src in logs:
        run  = os.path.basename(os.path.dirname(src))
        dstd = os.path.join(WORKING_DIR, '.metrics', run)
        os.makedirs(dstd, exist_ok=True)
        dst = os.path.join(dstd, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst); n += 1
    print(f'restored {n} files from previous session' if n else 'fresh session — nothing to restore')
    for lg in sorted(glob.glob(os.path.join(WORKING_DIR, '.metrics', '*', 'training_log.csv'))):
        n_ep = sum(1 for _ in open(lg)) - 1
        print(f'    resumed log: {os.path.basename(os.path.dirname(lg))} — {n_ep} epochs will be APPENDED to')
_restore()

## 3. Train/val split (90/10, seed 42 — INDEPENDENT of the training seed)

In [ ]:
import random
from pathlib import Path
SUBS=['low_blur_noise','high_sharp_scaled']
TRAIN_SRC=Path(LOLBLUR)/'train'; SPLIT=Path(WORKING_DIR)/'LOLBlur_split'
random.seed(42)   # split seed is FIXED across all stages/seeds — never change
clips=sorted([d.name for d in (TRAIN_SRC/'low_blur_noise').iterdir() if d.is_dir()])
n_val=max(1,int(len(clips)*0.1)); val=set(random.sample(clips,n_val)); tr=[c for c in clips if c not in val]
for nm,cl in [('physir_train',tr),('physir_val',val)]:
    for sub in SUBS:
        b=SPLIT/nm/sub; b.mkdir(parents=True,exist_ok=True)
        for c in cl:
            d=b/c
            if not d.exists(): d.symlink_to((TRAIN_SRC/sub/c).resolve())
PHYSIR_TRAIN=str(SPLIT/'physir_train'); PHYSIR_VAL=str(SPLIT/'physir_val')
print('train',len(tr),'val',len(val))

## 4. Runs — seeds x load-bearing arms, pair-major

In [ ]:
import yaml
EPOCHS   = 200        # identical to the seed-42 Set runs — never vary this for replicates
SNAPSHOT = 150        # frozen mid-run operating point
PERCEP_W = 0.01       # DarkIR's lambda_pe (Eq.9); LPIPS(VGG19) = their L_percep
BUDGET_H = 0          # 0 = no self-stop (train until session end); atomic saves make that safe

SEEDS = [2024]        # ← THE KNOB. [1337], or [1337, 2024] to do both here. Split stays 42.
SPLIT_SEED = 42       # never changes — the data split is fixed across all seeds/experiments

# arm -> (edge_on, phase_w, retinex_w, blur_w, percep_on); pair-major order below
ARMS = {
  'abl_baseline':    dict(edge=True, phase=0.0,  retinex=0.0,  blur=0.0, percep=False),
  'abl_full':        dict(edge=True, phase=0.20, retinex=0.18, blur=0.0, percep=False),
  'abl_percep':      dict(edge=True, phase=0.0,  retinex=0.0,  blur=0.0, percep=True),
  'abl_full_percep': dict(edge=True, phase=0.20, retinex=0.18, blur=0.0, percep=True),
}
# PAIR-MAJOR: complete each seed's arm sequence before moving to the next seed
CONFIGS = {f'{arm}_s{seed}': dict(sp, seed=seed)
           for seed in SEEDS for arm, sp in ARMS.items()}

def build_cfg(name, sp, budget_h=0):
    phys_on = (sp['phase'] > 0 or sp['retinex'] > 0 or sp['blur'] > 0)
    cfg = {
      'seed': sp['seed'],
      'datasets': {'name':'LOLBlur',
        'train': {'low_path':f'{PHYSIR_TRAIN}/low_blur_noise','high_path':f'{PHYSIR_TRAIN}/high_sharp_scaled',
                  'crop_size':256,'batch_size':4,'n_workers':4,'use_flips':True},
        'val': {'test_path':PHYSIR_VAL,'batch_size_test':1,'verbose':False,'n_workers':4}},
      'network': {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
                  'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True},
      'save': {'pretrained':None,'path':f'{WORKING_DIR}/{name}_latest.pt','best':f'{WORKING_DIR}/{name}_best.pt',
               'save_freq':2,'eval_freq':10,'perceptual_eval_epochs':[],
               'snapshot_epochs':[SNAPSHOT]},
      'pixel_criterion':'l1',
      'perceptual': sp['percep'], 'perceptual_weight':PERCEP_W,
      'perceptual_type':'lpips','perceptual_net':'vgg',
      'perceptual_criterion':'l1','perceptual_reduction':'mean',
      'edge': sp['edge'],'edge_weight':0.05,'edge_criterion':'l2','edge_reduction':'mean',
      'frequency':False,
      'enhance':True,'enhance_weight':0.01,'enhance_criterion':'l1','enhance_reduction':'mean',
      'physics_losses': {'enabled':phys_on,'phase2_start':9999,'phase3_start':9999,'phase4_start':1,
                         'retinex': sp['retinex']>0,'retinex_weight':sp['retinex'],
                         'blur': sp['blur']>0,'blur_weight':sp['blur'],
                         'phase': sp['phase']>0,'phase_weight':sp['phase']},
      'train': {'epochs':EPOCHS,'lr_initial':0.00028,'weight_decay':0.001,'lr_scheme':'CosineAnnealing',
                'eta_min':1e-6,'grad_clip':1.0,'time_budget_h':budget_h,
                'ema':{'enabled':True,'decay':0.9999,'warmup':2000}},
      'wandb':{'init':False,'name':name},'Resize':False,
    }
    p = f'{WORKING_DIR}/{name}.yml'
    yaml.dump(cfg, open(p,'w'), default_flow_style=False)
    return p, cfg

print('runs (pair-major):')
for n, sp in CONFIGS.items():
    print(f'  {n:24s} seed={sp["seed"]}  phase={sp["phase"]} retinex={sp["retinex"]} percep={sp["percep"]}')
print(f'\n{len(CONFIGS)} runs x ~28-35 h each on T4x2 — expect multiple sessions/quota cycles.')

## 5. TRAIN loop — idempotent + resume-safe

In [ ]:
import csv, subprocess
import torch

def done_ep(name):
    # Ground truth = the resume checkpoint itself (what train.py will actually resume from);
    # the training log is only a fallback if the checkpoint is missing/unreadable.
    ck = os.path.join(WORKING_DIR, f'{name}_latest.pt')
    if os.path.isfile(ck):
        try:
            return int(torch.load(ck, map_location='cpu', weights_only=False).get('epoch', 0))
        except Exception as e:
            print(f'  ({name}_latest.pt unreadable: {e} — falling back to log)')
    lg = Path(WORKING_DIR)/'.metrics'/name/'training_log.csv'
    if not lg.exists(): return 0
    last = 0
    try:
        for r in csv.DictReader(open(lg)): last = max(last, int(r['epoch']))
    except Exception: pass
    return last

def mirror_metrics():
    src = os.path.join(WORKING_DIR, '.metrics'); dst = os.path.join(WORKING_DIR, 'metrics_out')
    if os.path.isdir(src): shutil.copytree(src, dst, dirs_exist_ok=True)

# Runs until every arm reaches EPOCHS or Kaggle ends the session — no self-termination
# (BUDGET_H=0). Atomic 2-epoch checkpoints mean a kill at ANY moment loses ≤2 epochs and
# resumes seamlessly next session (weights, optimizer, scheduler, EMA, blur-σ, best-so-far).
for name, sp in CONFIGS.items():
    d = done_ep(name)
    if d >= EPOCHS:
        print(f'✅ {name}: trained {d}/{EPOCHS} ep — skip'); continue
    p, _ = build_cfg(name, sp, budget_h=BUDGET_H)
    tag = f'(resume ep {d})' if d > 0 else '(scratch)'
    print(f'\n===== TRAIN {name} {tag} → {EPOCHS} ep =====')
    r = subprocess.run(['torchrun','--nproc_per_node=2', f'{CODE_ROOT}/train.py', '-p', p])
    print(f'===== {name} exit {r.returncode} | now {done_ep(name)}/{EPOCHS} ep =====')
    mirror_metrics()
mirror_metrics()
print('\nTRAIN loop done —', {n: f'{done_ep(n)}/{EPOCHS}' for n in CONFIGS})

## 6. EVAL — LOLBlur test x {ep150, best, last} + per-image CSVs

In [ ]:
import torch, numpy as np, csv as _csv
import train as T
from data import create_test_data
device='cuda'
opt_test={'name':'LOLBlur','val':{'test_path':os.path.join(LOLBLUR,'test'),'batch_size_test':1},
          'train':{'n_workers':4,'verbose':True}}
tl_raw,_=create_test_data(rank=0,world_size=1,opt=opt_test)
tl = tl_raw if not isinstance(tl_raw,dict) else list(tl_raw.values())[0]['loader']
# clip-qualified image keys (LOLBlur test basenames repeat across its 30 clips), written to
# every row so a future ordering change is DETECTABLE instead of silent.
_ds = (tl if 'tl' in dir() else LOADER).dataset
FNAMES = [os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p)) for p in _ds.imgs_low]
assert len(set(FNAMES)) == len(FNAMES), 'image keys are not unique'
lpips_fn = T._LPIPS(net='vgg',verbose=False).to(device) if T._LPIPS_AVAILABLE else None
PERIMG=Path(WORKING_DIR)/'.metrics'/'per_image'; PERIMG.mkdir(parents=True,exist_ok=True)

net_cfg={'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
         'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

@torch.no_grad()
def eval_test(model, out_csv):
    rows=[]
    for i,(high,low) in enumerate(tl):
        high,low=high.to(device),low.to(device)
        pred=model(low,side_loss=False).clamp(0,1)
        _,_,H,W=high.shape; pred=pred[:,:,:H,:W]
        mse=torch.mean((pred-high)**2).item()
        psnr=20*np.log10(1.0/(np.sqrt(mse)+1e-8))
        T._ssim_metric.to(device); ssim=T._ssim_metric(pred,high).item()
        lp=lpips_fn(pred*2-1,high*2-1).mean().item()
        rows.append((i,FNAMES[i],psnr,ssim,lp))
    with open(out_csv,'w',newline='') as f:
        w=_csv.writer(f); w.writerow(['idx','fname','psnr','ssim','lpips']); w.writerows(rows)
    a=np.array([r[2:] for r in rows])
    return a.mean(0)

# same three operating points as the seed-42 runs, EMA weights
def ckpts(name):
    return [('ep150', f'{WORKING_DIR}/{name}_latest_ep{SNAPSHOT}.pt'),
            ('best',  f'{WORKING_DIR}/{name}_best.pt'),
            ('last',  f'{WORKING_DIR}/{name}_latest.pt')]

for name in CONFIGS:
    if done_ep(name) < EPOCHS:
        print(f'⏳ {name}: {done_ep(name)}/{EPOCHS} ep — finish training first'); continue
    for tag, ck_path in ckpts(name):
        csvp = PERIMG/f'per_image_{name}_{tag}.csv'
        if csvp.exists():
            print(f'✅ {name}[{tag}]: per-image exists — skip'); continue
        if not os.path.isfile(ck_path):
            print(f'⚠️ {name}[{tag}]: {os.path.basename(ck_path)} missing'); continue
        ck=torch.load(ck_path,map_location='cpu',weights_only=False)
        model=T.build_model(net_cfg).to(device)
        state=ck.get('ema') or ck['model']
        model.load_state_dict({k.replace('module.',''):v for k,v in state.items()},strict=False); model.eval()
        ps,ss,lp = eval_test(model,csvp)
        print(f'{name:24s}[{tag:5s}@ep{ck.get("epoch")}] PSNR={ps:.4f} SSIM={ss:.4f} LPIPS={lp:.4f}')
        del model; torch.cuda.empty_cache()
mirror_metrics()
print('\nEVAL loop complete.')

## 7. Leaderboard + per-seed physics delta

In [ ]:
import pandas as pd, glob as _g
rows=[]
for f in sorted(_g.glob(str(Path(WORKING_DIR)/'.metrics'/'per_image'/'per_image_*.csv'))):
    base=os.path.basename(f)[len('per_image_'):-len('.csv')]
    run,tag=base.rsplit('_',1)
    df=pd.read_csv(f)
    seed = run.rsplit('_s',1)[1] if '_s' in run else '42'
    arm  = run.rsplit('_s',1)[0] if '_s' in run else run
    rows.append({'arm':arm,'seed':seed,'ckpt':tag,'psnr':df.psnr.mean(),
                 'ssim':df.ssim.mean(),'lpips':df.lpips.mean(),'n':len(df)})
if rows:
    b=pd.DataFrame(rows).sort_values(['ckpt','arm','seed'])
    print('=== Seed replicates @256/200 — LOLBlur test ===')
    print(b.to_string(index=False))
    last=b[b.ckpt=='last']
    if {'abl_baseline','abl_full'} <= set(last.arm):
        for s in sorted(last.seed.unique()):
            sub=last[last.seed==s]
            if {'abl_baseline','abl_full'} <= set(sub.arm):
                d=(sub[sub.arm=='abl_full'].psnr.values[0]-sub[sub.arm=='abl_baseline'].psnr.values[0])
                print(f'  physics delta @seed {s}: {d:+.4f} dB')
        print('  (seed-42 reference from the Set runs: +1.59 dB)')
    print('\nper-image CSVs → metrics_out/per_image/ ; combine with seed-42 CSVs offline')
    print('for mean ± spread across seeds + per-seed paired Wilcoxon.')
else:
    print('no eval CSVs yet — run train + eval loops')